# Notebook for assignment Responsible AI predictive XAI exercise

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import torchvision
import pandas as pd
import numpy as np

from utils.notebook import display_scrollable_dataframe,plot_sailency
from data_loaders import CUB_extnded_dataset
from models import get_inception_transform
from IPython.display import display

from sailency import get_saliency_maps,saliency_score_part

In [ ]:
# Settings for the experiment your running
data_set = 'val' 

id = 533 # The id of the image you want to explain

# Settings for loading the dataset, make sure its the same as the one used to train the model. 
# You can find the settings in the .hydra folders config.yaml 
data_config = {'CUB_dir':r'data/CUB_200_2011',
                'split_file':r'data/train_test_val.pkl',
                'use_majority_voting':True,
                'min_class_count':10,
                'return_visibility':True}



In [ ]:
#Define data set, the human transformer data_set is used to get the original images instead of the normalized ones
transformer = get_inception_transform(mode='val',methode="center")
human_tansform = torchvision.transforms.Compose([torchvision.transforms.CenterCrop(299),torchvision.transforms.ToTensor()])

#Get dataset
data = CUB_extnded_dataset(mode=data_set,config_dict=data_config,transform=transformer)

#Get a dataset that return original images instead normalized ones
data_human = CUB_extnded_dataset(mode=data_set,config_dict=data_config,transform=human_tansform)
concept_names = data.consept_labels_names
class_names = data.class_labels_names
n_classes = data.n_classes
n_concepts = data.n_concepts

In [ ]:

X, C, Y,coordinates = data.__getitem__(id)

print("Image shape: ",X.shape)
print("Concept shape: ",C.shape)
print("Label shape: ",Y.shape)
X = X.unsqueeze(0)

img ,_,_,_ = data_human.__getitem__(id)

print(Y.argmax())

In [ ]:
# Separate the concepts and their visibility only relevant if visibility is used and no majority voting is used
if len(C.shape) == 2:
    Concepts = C[0]
    Concepts_visiblity = C[1]
else:
    Concepts = C.tolist()
    Concepts_visiblity = [None]*len(C)

In [ ]:
# Show the image and the coordinates of the concepts
plt.imshow(img.permute(1, 2, 0))

for i in range(len(coordinates)):
    if len(coordinates[i]) > 0:
        #print(f'coordinates {coordinates[i]} atribute_locations_names {consept_labels_masked[i]}')
        for x,y in coordinates[i]:
            plt.plot(x,y,'ro')
            name=concept_names[i].split("_")[1]
            if name == "forehead":
                ofset = -30

            elif name == "eye":
                ofset = 0
            else:
                ofset = 0
            
            plt.text(x+ofset,y,name,fontsize=9,color='red')
    plt.axis('off')
            
plt.size=(50,50)
plt.show()

# Expandability for Sequential and independent models.  

In [ ]:
#Load models make sure the model is trained with the same settings as the data loader
model_folder = r"models/Sequential_Basemodel3"

X_to_C_path = os.path.join(model_folder,"best_XtoC_model.pth")
C_to_Y_path = os.path.join(model_folder,"best_CtoY_model.pth")

ModelXtoC = torch.load(X_to_C_path,map_location=torch.device('cpu'))
ModelCtoY = torch.load(C_to_Y_path,map_location=torch.device('cpu'))

In [ ]:
#Make prediction

C_hat = ModelXtoC(X)
Y_hat = ModelCtoY(C_hat)

print(f"Predicted class: {Y_hat.argmax().item()} true class: {Y.argmax().item()} probability of true class: {Y_hat[0,Y.argmax()].item()}")

In [ ]:
#Make Concept prediction and other stuff
weights = ModelCtoY.linear.weight[Y.argmax().item()] #Find the weigths of the bird you guessed on.
Concept_frame = pd.DataFrame({"Concept":concept_names,"Concept_true":Concepts,"Concept_visiblity":Concepts_visiblity,"Concept_pred":np.round(C_hat[0].detach().numpy(),2),"CtoY_weight":weights.detach().numpy()})
display_scrollable_dataframe(Concept_frame)

In [ ]:
#List of concepts to plot
concept_list = [51,50,111]

sailency_maps = get_saliency_maps(X,concept_list,ModelXtoC,method_type="vanilla")

plot_sailency(img,sailency_maps,concept_list,concept_names,coordinates)

# Saliency map for Joint models  

In [ ]:
#Load models make sure the model is trained with the same settings as the data loader
model_path = r'models/Joint_Basemodel3/best_Joint_model.pth'
n_classes = data.n_classes
n_concepts = data.n_concepts

#Load the model
Joint_model = torch.load(model_path,map_location=torch.device('cpu'))




In [ ]:
#Make prediction
C_hat,Y_hat = Joint_model(X)
print(f"Predicted class: {Y_hat.argmax().item()} true class: {Y.argmax().item()} probability of true class: {Y_hat[0,Y.argmax()].item()}")

In [ ]:
#Make Concept prediction and other stuff
weights = Joint_model.MLP_model.linear.weight[Y.argmax().item()]
Concept_frame = pd.DataFrame({"Concept":concept_names,"Concept_true":Concepts,"Concept_visiblity":Concepts_visiblity,"Concept_pred":np.round(C_hat[0].detach().numpy(),2),"CtoY_weight":weights.detach().numpy()})
display_scrollable_dataframe(Concept_frame)

In [ ]:
concept_list = [51,50,111]

#Make model only predict the concept C
Joint_model.set_sailency_output("C")

sailency_maps = get_saliency_maps(X,concept_list,Joint_model,method_type="vanilla")

plot_sailency(img,sailency_maps,concept_list,concept_names,coordinates)

In [ ]:
concept_list = [51,50,111]

#Make model only predict the concept C
Joint_model.set_sailency_output("Y")

sailency_maps = get_saliency_maps(X,concept_list,Joint_model,method_type="vanilla")

plot_sailency(img,sailency_maps,concept_list,class_names)